In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH = "/content/Churn_Modelling.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.info())
print("\nChurn distribution:\n", df["Exited"].value_counts(normalize=True))


Shape: (10000, 14)
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63     

In [ ]:
drop_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
df = df.drop(columns=drop_cols)

# Check for missing values
print("\nMissing values:\n", df.isnull().sum())


Missing values:
 CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="Exited", data=df)
plt.title("Churn Distribution (0 = Stayed, 1 = Churned)")
plt.tight_layout()
plt.savefig("eda_churn_distribution.png")
plt.close()

plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png")
plt.close()


In [ ]:
X = df.drop(columns=["Exited"])
y = df["Exited"]

categorical_features = ["Geography", "Gender"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=42, class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    ),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



===== Logistic Regression =====
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.70      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000

Confusion Matrix:
 [[1143  450]
 [ 122  285]]

===== Random Forest =====
              precision    recall  f1-score   support

           0       0.92      0.86      0.89      1593
           1       0.56      0.72      0.63       407

    accuracy                           0.83      2000
   macro avg       0.74      0.79      0.76      2000
weighted avg       0.85      0.83      0.84      2000

Confusion Matrix:
 [[1364  229]
 [ 114  293]]

===== Gradient Boosting =====
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.80      0.50      0.61       407


In [ ]:
results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
print("\n===== Model Comparison =====")
print(results_df.to_string(index=False))
results_df.to_csv("model_comparison.csv", index=False)



===== Model Comparison =====
              Model  Accuracy  Precision   Recall       F1  ROC-AUC
  Gradient Boosting    0.8720   0.798419 0.496314 0.612121 0.870911
      Random Forest    0.8285   0.561303 0.719902 0.630786 0.865985
Logistic Regression    0.7140   0.387755 0.700246 0.499124 0.777182


In [ ]:
plt.figure(figsize=(7, 6))
for name, pipe in fitted_pipelines.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png")
plt.close()


In [ ]:
best_tree_model_name = "Random Forest" if "Random Forest" in fitted_pipelines else None
if best_tree_model_name:
    pipe = fitted_pipelines[best_tree_model_name]
    ohe_cols = pipe.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features)
    all_feature_names = numeric_features + list(ohe_cols)
    importances = pipe.named_steps["classifier"].feature_importances_

    fi_df = pd.DataFrame({"Feature": all_feature_names, "Importance": importances})
    fi_df = fi_df.sort_values("Importance", ascending=False)

    plt.figure(figsize=(8, 6))
    sns.barplot(x="Importance", y="Feature", data=fi_df)
    plt.title(f"Feature Importance ({best_tree_model_name})")
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    plt.close()

    print("\nTop features driving churn:\n", fi_df.head(10).to_string(index=False))

print("\nDone. Outputs saved: eda_churn_distribution.png, eda_correlation_heatmap.png, "
      "roc_curves.png, feature_importance.png, model_comparison.csv")



Top features driving churn:
           Feature  Importance
              Age    0.358202
    NumOfProducts    0.248108
          Balance    0.101834
Geography_Germany    0.063428
   IsActiveMember    0.061442
  EstimatedSalary    0.051883
      CreditScore    0.051596
           Tenure    0.027379
      Gender_Male    0.022856
  Geography_Spain    0.007000

Done. Outputs saved: eda_churn_distribution.png, eda_correlation_heatmap.png, roc_curves.png, feature_importance.png, model_comparison.csv
